In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Loading Gold Macro Table for Advanced Analytics...")

# 1. Read the gold macro table
df_macro = spark.read.table("gold_public_safety_macro")

# 2. Window specification to calculate state averages and metrics
state_window = Window.partitionBy("state_code")

# 3. Computing Advanced Analytics (Z-Score, IVSC, and State Averages)
df_analytics = df_macro \
    .withColumn("state_avg_crime", F.avg("total_occurrences").over(state_window)) \
    .withColumn("state_std_crime", F.stddev("total_occurrences").over(state_window)) \
    .withColumn(
        "crime_z_score",
        F.when(
            (F.col("state_std_crime").isNotNull()) & (F.col("state_std_crime") > 0),
            F.round((F.col("total_occurrences") - F.col("state_avg_crime")) / F.col("state_std_crime"), 2)
        ).otherwise(0.0)
    ) \
    .withColumn(
        "inv_idh", F.lit(1.0) - F.col("idh_composite")
    ) \
    .withColumn(
        "ivsc_score",
        F.round((F.col("crime_rate_per_100k") * 0.4) + (F.col("inv_idh") * 0.6), 4)
    ) \
    .withColumn(
        "economic_tier",
        F.when(F.col("gdp_per_capita") > 50000, "High Economy")
         .when(F.col("gdp_per_capita") > 25000, "Medium Economy")
         .otherwise("Emerging Economy")
    )

# 4. Drop the existing table explicitly to avoid schema/metadata mismatch
table_name = "gold_advanced_analytics_intelligence"
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# 5. Materializing the Advanced Analytics Intelligence Table cleanly
df_analytics.write.format("delta").mode("overwrite").saveAsTable(table_name)

print("ADVANCED ANALYTICS SUCCESS! Intelligence table materialized successfully.")

display(df_analytics.select("municipality_name_clean", "state_code", "total_occurrences", "crime_z_score", "ivsc_score", "economic_tier").limit(10))